- ingest data from volume to delta table
- add metadata column such as file_name , ingest_timestamp

In [0]:
%run ../00-common/01.environment_config

In [0]:
%run ../00-common/02.bronze_helpers

In [0]:
source_file_path = f"{landing_file_path}/drivers.json"
full_table_name = f"{catalog_name}.{bronze_schema}.driver"

In [0]:
from pyspark.sql.types import StructField,StructType,DoubleType,StringType,IntegerType,DateType
# for production scenarion we define schema explicitly if new data is also come then also no issue occur, for development purpose we can used .option("inferSchema","true") this option
 
name_schema = StructType([
    StructField("givenName",StringType()),
    StructField("familyName",StringType())
])

driver_schema = StructType([
    StructField("driverId", StringType()),
    StructField("name", name_schema),
    StructField("dateOfBirth", DateType()),
    StructField("nationality", StringType()),
    StructField("url", StringType())
])

In [0]:
df_driver = spark.read.format("json") \
            .option("header","true") \
            .schema(driver_schema) \
            .load(source_file_path)

display(df_driver)


In [0]:
from pyspark.sql import functions as F

df_driver_final = add_ingest_metadata(df_driver)
display(df_driver_final)


In [0]:
df_driver_final.write.format("delta") \
                        .mode("overwrite") \
                        .saveAsTable(full_table_name)

In [0]:
display(spark.table(full_table_name))